# D4 Transportation Workflow Notebook

This notebook compares baseline and intervention mobility performance in sectioned simple and complex tracks.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen
from typing import cast

import matplotlib.pyplot as plt

OUTPUT_DIR = Path.cwd() / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALLOW_LIVE_API = os.getenv('GEOPROMPT_ALLOW_LIVE_API', '0') == '1'


def fetch_json(url: str, fallback: dict) -> dict:
    if not ALLOW_LIVE_API:
        return fallback
    try:
        req = Request(url, headers={'User-Agent': 'geoprompt-section-d-notebook/2.0'})
        with urlopen(req, timeout=10) as response:  # nosec B310
            return json.loads(response.read().decode('utf-8'))
    except (URLError, TimeoutError, ValueError):
        return fallback


import geoprompt as gp
from geoprompt import GeoPromptFrame, frame_to_geojson, write_geojson
from geoprompt.network.core import NetworkEdge
from geoprompt.network.routing import build_network_graph, shortest_path, service_area
from geoprompt.tools import build_scenario_report, export_scenario_report

## Section A: Pull Transportation Data Sources


In [2]:
overpass = fetch_json('https://overpass-api.de/api/interpreter?data=[out:json];way[%22highway%22](40.70,-111.95,40.80,-111.80);(._;>;);out body;', {'elements': []})
gtfs = fetch_json('https://transit.land/api/v2/rest/feeds', {'feeds': []})
incidents = fetch_json('https://data.transportation.gov/api/views/4f3n-jbg2/rows.json?accessType=DOWNLOAD', {'data': []})
print('Overpass elements:', len(overpass.get('elements', [])))
print('GTFS feeds:', len(gtfs.get('feeds', [])))
print('Incident rows:', len(incidents.get('data', [])))

Overpass elements: 0
GTFS feeds: 0
Incident rows: 0


## Section B: Simple Track


In [ ]:
edges: list[NetworkEdge] = [
    cast(NetworkEdge, {'edge_id': 'r1', 'from_node': 'O', 'to_node': 'A', 'cost': 5.0}),
    cast(NetworkEdge, {'edge_id': 'r2', 'from_node': 'A', 'to_node': 'B', 'cost': 4.0}),
    cast(NetworkEdge, {'edge_id': 'r3', 'from_node': 'B', 'to_node': 'C', 'cost': 3.0}),
    cast(NetworkEdge, {'edge_id': 'r4', 'from_node': 'O', 'to_node': 'C', 'cost': 15.0}),
    cast(NetworkEdge, {'edge_id': 'r5', 'from_node': 'B', 'to_node': 'D', 'cost': 5.0}),
    cast(NetworkEdge, {'edge_id': 'r6', 'from_node': 'D', 'to_node': 'E', 'cost': 2.0}),
    cast(NetworkEdge, {'edge_id': 'r7', 'from_node': 'E', 'to_node': 'C', 'cost': 2.0}),
]
graph = build_network_graph(edges, directed=False)
path_result = shortest_path(graph, 'O', 'C')
area = service_area(graph, origins=['O'], max_cost=10.0)

path_nodes: list[str] = path_result.get('path_nodes', [])
print(f"Shortest path O→C: {' → '.join(path_nodes)}  cost={path_result.get('total_cost', '?')}")

# Node coordinates for visualization (schematic x/y).
coords: dict[str, tuple[float, float]] = {
    'O': (0.0, 2.0), 'A': (1.0, 2.0), 'B': (2.0, 2.0),
    'C': (4.0, 2.0), 'D': (2.0, 1.0), 'E': (3.0, 1.0),
}

# Build GeoPromptFrame from service area results (use x/y as geometry coords).
area_rows = [
    {
        'node': str(r['node']),
        'cost': float(r['cost']),
        'geometry': {
            'type': 'Point',
            'coordinates': [coords.get(str(r['node']), (0.0, 0.0))[0],
                            coords.get(str(r['node']), (0.0, 0.0))[1]],
        },
    }
    for r in area
]
area_frame = GeoPromptFrame(area_rows, geometry_column='geometry')

# Spatial analysis: nearest neighbors in the service area graph.
if len(area_frame) > 1:
    neighbors = area_frame.nearest_neighbors(id_column='node', k=1)
    print('\n--- Nearest neighbors in service area ---')
    for nb in neighbors:
        print(f"  {nb['origin']} → {nb['neighbor']}  dist={nb['distance']:.3f}")

# Write GeoJSON using GeoPrompt.
geojson_path = OUTPUT_DIR / 'd4-notebook-network-map.geojson'
write_geojson(geojson_path, area_frame)
print(f'\nWrote GeoJSON map: {geojson_path}')
print(f'Service area: {len(area_frame)} reachable nodes')

# Show area frame summary.
print(json.dumps(area_frame.summary(), indent=2, default=str))

# ── Inline network visualization ──────────────────────────────────────────────
records = area_frame.to_records()
node_costs = {str(r['node']): float(r['cost']) for r in records}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Schematic node map with edges.
for e in edges:
    fn, tn = str(e['from_node']), str(e['to_node'])
    if fn in coords and tn in coords:
        axes[0].plot([coords[fn][0], coords[tn][0]], [coords[fn][1], coords[tn][1]],
                     'b-', alpha=0.3, linewidth=1.5)
for node, (x, y) in coords.items():
    color = '#27ae60' if node in node_costs else '#adb5bd'
    cost_label = f"\n(cost={node_costs[node]:.1f})" if node in node_costs else ''
    axes[0].scatter([x], [y], c=color, s=200, edgecolors='#333', zorder=5)
    axes[0].annotate(f'{node}{cost_label}', (x, y), textcoords='offset points',
                     xytext=(5, 5), fontsize=9)
for i in range(len(path_nodes) - 1):
    fn, tn = path_nodes[i], path_nodes[i + 1]
    if fn in coords and tn in coords:
        axes[0].plot([coords[fn][0], coords[tn][0]], [coords[fn][1], coords[tn][1]],
                     'r-', linewidth=3, label='Shortest path' if i == 0 else '')
axes[0].set_title(f'D4 Network — Shortest path: {" → ".join(path_nodes)}')
if path_nodes:
    axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')

# Travel cost bar chart.
node_names = sorted(node_costs.keys(), key=lambda n: node_costs[n])
costs = [node_costs[n] for n in node_names]
bar_colors = ['#27ae60' if c < 7 else '#e67e22' if c < 12 else '#c0392b' for c in costs]
axes[1].barh(node_names, costs, color=bar_colors)
axes[1].set_xlabel('Travel Cost from Origin O')
axes[1].set_title('Service Area — Travel Costs')
axes[1].axvline(10.0, color='#555', linestyle='--', alpha=0.7, label='Max cost=10')
axes[1].legend()
axes[1].grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

simple_payload = {
    'track': 'simple',
    'shortest_path': path_result,
    'service_area': area,
    'map_geojson': str(geojson_path),
}
(OUTPUT_DIR / 'd4-notebook-simple.json').write_text(json.dumps(simple_payload, indent=2, default=str), encoding='utf-8')
print('Wrote d4-notebook-simple.json')

,node,assigned_origin,cost,within_service_area
0,O,O,0.0,True
1,A,O,5.0,True
2,B,O,9.0,True


Shortest path {'origin': 'O', 'destination': 'C', 'reachable': True, 'total_cost': 12.0, 'path_nodes': ['O', 'A', 'B', 'C'], 'path_edges': ['r1', 'r2', 'r3'], 'hop_count': 3}
GeoPrompt parsed map features: 3
Wrote D:\Github\geoprompt\outputs\d4-notebook-simple.json
Wrote D:\Github\geoprompt\outputs\d4-notebook-network-map.html
Wrote D:\Github\geoprompt\outputs\d4-notebook-network-map.geojson


## Section C: Complex Track


In [ ]:
scenarios = {
    'baseline':        {'avg_travel_time_min': 34.0, 'reliability_index': 0.62, 'equity_gap': 0.31},
    'bus_lane':        {'avg_travel_time_min': 26.0, 'reliability_index': 0.75, 'equity_gap': 0.24},
    'signal_priority': {'avg_travel_time_min': 28.0, 'reliability_index': 0.71, 'equity_gap': 0.22},
}
report = build_scenario_report(scenarios['baseline'], scenarios['bus_lane'], higher_is_better=['reliability_index'])
report_path = export_scenario_report(report, OUTPUT_DIR / 'd4-notebook-scenario-report.json')
print('Scenario report:', report_path)

# Compute scenario scores using plain dicts.
ranking_rows = []
for name, vals in scenarios.items():
    score = round(
        (1.0 / vals['avg_travel_time_min']) * 35
        + vals['reliability_index'] * 45
        + (1.0 - vals['equity_gap']) * 20,
        4,
    )
    ranking_rows.append({
        'scenario': name,
        'travel_time': vals['avg_travel_time_min'],
        'reliability': vals['reliability_index'],
        'equity_gap': vals['equity_gap'],
        'score': score,
    })
ranking_rows.sort(key=lambda r: -float(r['score']))
print('\n--- Scenario Ranking ---')
for r in ranking_rows:
    print(f"  {r['scenario']:20s}  score={r['score']:.4f}  travel_time={r['travel_time']:.1f}m  reliability={r['reliability']:.2f}")

# ── Inline scenario comparison chart ─────────────────────────────────────────
names = [r['scenario'] for r in ranking_rows]
scores = [float(r['score']) for r in ranking_rows]
travel_times = [float(r['travel_time']) for r in ranking_rows]
reliabilities = [float(r['reliability']) for r in ranking_rows]
equity_gaps = [float(r['equity_gap']) for r in ranking_rows]

colors = ['#27ae60', '#2563eb', '#e67e22']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].barh(names, scores, color=colors)
axes[0].set_xlabel('Composite Score')
axes[0].set_title('Composite Score (higher = better)')
axes[0].grid(True, axis='x', alpha=0.3)

axes[1].barh(names, travel_times, color=colors)
axes[1].set_xlabel('Avg Travel Time (min)')
axes[1].set_title('Travel Time (lower = better)')
axes[1].grid(True, axis='x', alpha=0.3)

axes[2].barh(names, reliabilities, color=colors)
axes[2].set_xlabel('Reliability Index')
axes[2].set_title('Reliability Index (higher = better)')
axes[2].grid(True, axis='x', alpha=0.3)

plt.suptitle('D4 Transportation — Scenario Comparison', fontweight='bold')
plt.tight_layout()
plt.show()

complex_payload = {
    'track': 'complex',
    'ranking': ranking_rows,
    'scenario_report_path': str(report_path),
}
(OUTPUT_DIR / 'd4-notebook-complex.json').write_text(json.dumps(complex_payload, indent=2, default=str), encoding='utf-8')
print('Wrote d4-notebook-complex.json')

,scenario,travel_time,reliability,equity_gap,score
1,bus_lane,26.0,0.75,0.24,50.2962
2,signal_priority,28.0,0.71,0.22,48.8000
0,baseline,34.0,0.62,0.31,42.7294


Wrote D:\Github\geoprompt\outputs\d4-notebook-complex.json
